# Functions for Checking a Machine Learning Model


Two questions:
* Question 1: Do physical traits give an edge? This predicts `win_pct` which is the win percentage number, so this needs a regression model.
* Question 2: Does a fighter's body shape how they win? This predicts `win_style` which is the type of win such as KO, submission, or decision This is a category, so it needs a classification** model.

In [2]:
%pip install matplotlib scikit-learn

  Using cached matplotlib-3.11.2-cp315-cp315-macosx_11_0_arm64.whl.metadata (80 kB)
  Using cached scikit_learn-1.9.1-cp315-cp315-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached contourpy-1.4.0-cp315-cp315-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.68.0-cp315-cp315-macosx_10_15_universal2.whl.metadata (139 kB)
  Using cached kiwisolver-1.5.1-cp315-cp315-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached pillow-13.0.0-cp315-cp315-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached scipy-1.18.1-cp315-cp315-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached matplotlib-3.11.2-cp315-cp315-macosx_11_0_arm64.whl (9.3 MB)
Using cached scikit_learn-1.9.1-cp315-cp315-macosx_12_0_arm64.whl (8.3 MB)
Using cached contourpy-1.4.0-cp315-cp315-macosx_11_0_arm64.whl (283 kB)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.utils import shuffle

df = pd.read_csv("ufc_fighters_clean.csv")
print(df.shape)

(58, 18)


### Function 1: `check_regression_fit()` for Question 1

This function takes the model's predictions and fighters' win percentages and prints:
* R²
* Mean absolute error
* Baseline MAE

This function makes two plots: predicted vs actual, and the residuals which is actual minus predicted.

**Why this is useful:** R² is how much of the variation in win percentage the model explains. MAE is how far off the predictions are on average. Baseline MAE is how far off we'd be if we just guessed the average win percentage for every fighter. MAE by itself can be misleading. Most fighters in our dataset win more than they lose. Win percentages are all clustered together, so a model could get a small MAE just by guessing close to the average without really anything. If the model can't beat the baseline, then reach, height, and weight aren't actually helping it predict winning.


The residual plot checks whether a straight line is the right kind of model. If the residuals are randomly scattered around 0, the model is good. Otherwise, the model is missing something.

In [ ]:
def check_regression_fit(y_test, y_pred, baseline):
    """Print R^2, MAE, and a baseline MAE, then plot predicted vs actual and the residuals."""
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    baseline_guesses = [baseline] * len(y_test)
    baseline_mae = mean_absolute_error(y_test, baseline_guesses)

    print("R^2:", round(r2, 3))
    print("MAE:", round(mae, 3))
    print("Baseline MAE:", round(baseline_mae, 3))

    if mae < baseline_mae:
        print("The model beats the baseline.")
    else:
        print("The model does NOT beat the baseline, so the features aren't helping.")

    residuals = y_test - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].scatter(y_test, y_pred)
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red")
    axes[0].set_xlabel("actual value")
    axes[0].set_ylabel("predicted value")
    axes[0].set_title("Predicted vs. actual")
    axes[1].scatter(y_pred, residuals)
    axes[1].axhline(0, color="red")
    axes[1].set_xlabel("predicted value")
    axes[1].set_ylabel("residual (actual - predicted)")
    axes[1].set_title("Residuals")

    plt.tight_layout()
    plt.show()

### Function 2: `check_classifier()` for Question 2

The function prints the model’s accuracy, two baselines which is guessing the most common win style, and random guessing, the percent of correctness for each win style, and a confusion matrix.

**Why this is useful:**

1. Unbalanced classes: A lot of the fighters of our data are KO fighters. So always guessing “KO” beats random guessing. The model has to beat this baseline.
2. Ignored classes: Checking each style separately catches a model that never predicts one win style.


In [10]:
def check_classifier(y_test, y_pred, classes):
    """Print accuracy vs. two baselines, the percent correct for each class, and a confusion matrix."""
    y_test = list(y_test)
    y_pred = list(y_pred)

    correct = 0
    for i in range(len(y_test)):
        if y_test[i] == y_pred[i]:
            correct += 1
    accuracy = correct / len(y_test)

    most_common = pd.Series(y_test).value_counts().idxmax()
    majority_baseline = y_test.count(most_common) / len(y_test)

    random_baseline = 1 / len(classes)

    print("Accuracy:", round(accuracy * 100, 1), "%")
    print("Majority baseline (always guessing " + most_common + "):", round(majority_baseline * 100, 1), "%")
    print("Random guess baseline:", round(random_baseline * 100, 1), "%")

    if accuracy > majority_baseline:
        print("The model beats always guessing " + most_common + ".")
    else:
        print("The model does NOT beat always guessing " + most_common + ".")

    print()
    print("Percent correct for each win style:")
    for c in classes:
        total = y_test.count(c)
        right = 0
        for i in range(len(y_test)):
            if y_test[i] == c and y_pred[i] == c:
                right += 1
        if total == 0:
            print(" ", c, ": no fighters with this win style in the test set")
        else:
            print(" ", c, ":", round(right / total * 100, 1), "% (", right, "of", total, ")")
            if right == 0:
                print("    warning: the model never gets", c, "right")

    print()
    print(pd.crosstab(pd.Series(y_test, name="actual"), pd.Series(y_pred, name="predicted")))

### Function 3: `check_stability()` for Both Questions

This functions uses k-fold cross validation to test the model on the data. Then it prints each fold’s training score, test score, their average, and standard deviation.

**Why this is useful:**

One train/test split can mean lucky or unlucky, a big standard deviation means no single score can be trusted, and a training score much higher than the test scores means the model is memorizing fighters. If both big standard deviation and training score are low, the features are not related to the target.


In [11]:
def check_stability(model, X, y, k=5):
    """Use k-fold cross validation to check if the score is stable and if the model is overfitting."""
    # shuffle first since the data is sorted by weight class
    X, y = shuffle(X, y, random_state=3000)

    test_scores = cross_val_score(model, X, y, cv=k)

    model.fit(X, y)
    train_score = model.score(X, y)

    print("Test score for each fold:", test_scores.round(3))
    print("Average test score:", round(test_scores.mean(), 3))
    print("Standard deviation:", round(test_scores.std(), 3))
    print("Training score:", round(train_score, 3))

    if test_scores.std() > 0.1:
        print("The scores change a lot between folds, so one train/test split could be misleading.")
    if train_score - test_scores.mean() > 0.15:
        if train_score < 0.2:
            print("The model doesn't even fit the training data well, so the features might just not be related to the target.")
        else:
            print("The model does a lot better on the training data, so it might be overfitting.")